# Model Comparison — Predictive Maintenance

**Phase 8.2** — Compare all models: bearing fault (feature + raw) and RUL (NASA C-MAPSS).

## 1. Bearing fault models (CWRU)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.load_data import DATA_DIR
from src.feature_engineering import build_dataset, build_raw_dataset
from src.predict import (
    load_model_and_meta,
    load_raw_model_and_meta,
    predict_from_file,
    predict_from_file_raw,
)

In [ ]:
# Feature-based model
MODEL_PATH = ROOT / "models" / "fault_classifier.keras"
RAW_PATH = ROOT / "models" / "fault_classifier_raw.keras"

models_ok = []
if MODEL_PATH.exists():
    model, mean, std, names = load_model_and_meta(MODEL_PATH)
    print(f"Feature model: {len(names)} classes — {names}")
    models_ok.append("feature")
else:
    print("Feature model not found. Run: python scripts/train.py")

if RAW_PATH.exists():
    model, mean, std, names, win = load_raw_model_and_meta(RAW_PATH)
    print(f"Raw model: window={win}, {len(names)} classes")
    models_ok.append("raw")
else:
    print("Raw model not found. Run: python scripts/train_raw.py --arch 1dcnn")

In [ ]:
# Quick prediction on sample files
mat_files = list((ROOT / "data").glob("*.mat"))
if mat_files:
    sample = mat_files[0]
    print(f"Sample: {sample.name}")
    if "feature" in models_ok:
        r = predict_from_file(sample)
        print(f"  Feature → {r['predicted_class']} ({r['probability']:.1%})")
    if "raw" in models_ok:
        r = predict_from_file_raw(sample)
        print(f"  Raw    → {r['predicted_class']} ({r['probability']:.1%})")

## 2. RUL model (NASA C-MAPSS)

In [ ]:
from src.load_cmapss import load_fd001, CMAPSS_DIR
from src.predict import load_rul_model_and_meta
import numpy as np

RUL_PATH = ROOT / "models" / "rul_predictor.keras"

if RUL_PATH.exists() and (CMAPSS_DIR / "test_FD001.txt").exists():
    model, scaler_min, scaler_max, win, n_feat, used_cols = load_rul_model_and_meta(RUL_PATH)
    train_df, test_df, true_rul = load_fd001(CMAPSS_DIR, fd=1)
    used_cols = list(used_cols)
    preds = []
    for i, unit in enumerate(test_df["unit"].unique()[:10]):
        u = test_df[test_df["unit"] == unit].sort_values("cycle")
        arr = u.iloc[:, used_cols].values.astype(np.float32)
        if len(arr) >= win:
            w = arr[-win:]
            scale = scaler_max - scaler_min
            scale = np.where(scale < 1e-10, 1.0, scale)
            w = (w - scaler_min) / scale * 2.0 - 1.0
            rul = float(model.predict(w[np.newaxis, ...], verbose=0)[0, 0])
            rul = max(0, rul)
            true = int(true_rul[i]) if i < len(true_rul) else "—"
            preds.append((unit, rul, true))
    print("Engine | Pred RUL | True RUL")
    for u, p, t in preds:
        print(f"  {u:3d}  | {p:7.1f} | {t}")
else:
    print("RUL model or data not found. Run: python scripts/download_cmapss.py && python scripts/train_rul.py")

## 3. Summary

In [ ]:
print("""
Models in this project:
─────────────────────
• Feature-based (Dense): 9 features → 4 classes. Val acc ~99–100%.
• Raw-signal (1D-CNN): 1024-sample windows → 4 classes. Val acc ~99.9%.
• RUL (LSTM): 30-cycle sensor windows → cycles until failure. Val RMSE ~15–25.

Datasets: CWRU (bearings), NASA C-MAPSS FD001 (turbofan).
""")